# Тренажёр определения времени — демонстрация

Бекмуратов Дилмурат Бекмуратович, 231-321.

Показывает работу `libClockCore` напрямую из кода, без UI, а также примеры запуска консольного и WinForms клиентов через параметры командной строки.

> Перед запуском ячеек собери библиотеку: `dotnet build ../libClockCore/libClockCore.csproj`

In [ ]:
#r "../libClockCore/bin/Debug/net10.0/libClockCore.dll"
using ClockCore;

## Упражнение «выставь стрелки» (SetTimeExercise)

In [ ]:
var generator = new RandomExerciseGenerator(new Random(42));
var setTime = (SetTimeExercise)generator.Next(ExerciseKind.SetTime, Difficulty.Medium);

Console.WriteLine(setTime.Prompt);

var userAnswer = setTime.Target; // имитируем верный ответ
var isCorrect = setTime.CheckAnswer(userAnswer);

Console.WriteLine($"Ответ пользователя: {userAnswer:HH:mm} -> {(isCorrect ? "верно" : "неверно")}");

## Упражнение «сколько времени прошло» (ElapsedTimeExercise)

In [ ]:
var elapsed = (ElapsedTimeExercise)generator.Next(ExerciseKind.ElapsedTime, Difficulty.Hard);

Console.WriteLine(elapsed.Prompt);
Console.WriteLine($"Ожидаемый ответ: {elapsed.CorrectAnswerText}");

var wrongAnswer = elapsed.Expected.Add(TimeSpan.FromMinutes(5));
Console.WriteLine($"Проверка заведомо неверного ответа: {elapsed.CheckAnswer(wrongAnswer)}");

## TrainerSession — несколько раундов с событиями и статистикой

In [ ]:
var session = new TrainerSession(new RandomExerciseGenerator(new Random(7)), TrainerMode.Test, ExerciseKind.SetTime, Difficulty.Easy);

session.ExerciseGenerated += (_, ex) => Console.WriteLine($"Новое упражнение: {ex.Prompt}");
session.AnswerEvaluated += (_, correct) => Console.WriteLine(correct ? "-> верно" : "-> неверно");

for (int i = 0; i < 3; i++)
{
    session.NewExercise();
    var target = ((SetTimeExercise)session.Current!).Target;
    session.Submit(i == 1 ? target.AddMinutes(30) : target); // один раунд специально с ошибкой
}

Console.WriteLine($"Итог: верно {session.Stats.Correct} из {session.Stats.Total}, точность {session.Stats.Accuracy:F0}%");

## Запуск клиентов из командной строки

```bash
# справка
dotnet run --project ../cnsClockTrainer/cnsClockTrainer.csproj -- --help

# консольный тренажёр: проверка знаний, "сколько прошло", сложный уровень, 5 раундов
dotnet run --project ../cnsClockTrainer/cnsClockTrainer.csproj -- --mode=test --exercise=elapsed --difficulty=hard --rounds=5

# WinForms-клиент сразу в режиме обучения
dotnet run --project ../wfaClockTrainer/wfaClockTrainer.csproj -- --mode=learn --exercise=set --difficulty=easy
```